# Load Data

In [1]:
!pip install pandas
!pip install fastparquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.2 MB/s  0:00:01m 1.2 MB/s eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.3 MB/s  0:00:002.4 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]━━━━━━ 2/3 [fastparquet]


In [2]:
import pandas as pd

all_pr_df = pd.read_parquet("Data/all_pull_request.parquet")
human_pr_df_without_bots = pd.read_parquet("Data/human_pull_request_removed_bot_prs.parquet")
human_repos_urls_df_removed = pd.DataFrame({'repo_url' : human_pr_df_without_bots['repo_url'].unique()})
all_prs_from_repos_with_human_prs = all_pr_df.merge(human_repos_urls_df_removed, on='repo_url')

pr_comments_df = pd.read_parquet("Data/pr_comments.parquet")
pr_comments_from_repos_with_human_df = pr_comments_df[
    pr_comments_df['pr_id'].isin(all_prs_from_repos_with_human_prs['id'])
]

human_pr_comments_df = pd.read_parquet("Data/human_pr_comments.parquet")
human_pr_comments_from_repos_with_human_df = human_pr_comments_df[
    human_pr_comments_df['pr_id'].isin(human_pr_df_without_bots['id'])
]

# Prepare Data

## BotOther

### pr_comments

In [3]:
# Loading pr_comments.parquet
pr_comments_df = pd.read_parquet("Data/pr_comments.parquet")

# Filter user_type=="User"
user_filter = pr_comments_df['user_type'] == "User"

# Filter potential bots by username
bot_like_filter = pr_comments_df['user'].str.contains(r'bot|ci|copilot|ai|automation', case=False, na=False)

# Combine filters
suspected_bots = pr_comments_df[user_filter & bot_like_filter]

# Count the number of comments for each user
bot_counts = suspected_bots['user'].value_counts()

# Filter those who have a minimum of 10 comments
bot_counts_10plus = bot_counts[bot_counts >= 10]

# Sort records by comment count
bot_counts_10plus_sorted = bot_counts_10plus.sort_values(ascending=False)

# Display the first 50 suspects
display(bot_counts_10plus.head(50))

# Find Bots and change to BotOther
bots_users = ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot', 'facebook-github-bot','pulumi-bot','raycastbot']
pr_comments_df.loc[pr_comments_df['user'].isin(bots_users), 'user_type'] = 'BotOther'

user
captainsafia       52
jakebailey         48
ian-at-airbyte     34
NoritakaIkeda      32
k8s-ci-robot       28
MayaRainer         27
wolfSSL-Bot        25
vdaas-ci           25
jairmyree          15
hairyhum           13
raineorshine       13
takenagain         12
pytorchmergebot    11
aishwaryabh        11
owidbot            10
Name: count, dtype: int64

### pr_comments_from_repos_with_human_df

In [4]:
# Filter user_type=="User"
pr_comments_from_repos_with_human_df_user_filter = pr_comments_from_repos_with_human_df['user_type'] == "User"

# Filter potential bots by username
pr_comments_from_repos_with_human_df_bot_like_filter = pr_comments_from_repos_with_human_df['user'].str.contains(r'bot|ci|copilot|ai|automation', case=False, na=False)

# Combine filters
pr_comments_from_repos_with_human_df_suspected_bots = pr_comments_from_repos_with_human_df[pr_comments_from_repos_with_human_df_user_filter & pr_comments_from_repos_with_human_df_bot_like_filter]

# Count the number of comments for each user
pr_comments_from_repos_with_human_df_bot_counts = pr_comments_from_repos_with_human_df_suspected_bots['user'].value_counts()

# Filter those who have a minimum of 10 comments
pr_comments_from_repos_with_human_df_bot_counts_10plus = pr_comments_from_repos_with_human_df_bot_counts[pr_comments_from_repos_with_human_df_bot_counts >= 10]

# Sort records by comment count
pr_comments_from_repos_with_human_df_bot_counts_10plus_sorted = pr_comments_from_repos_with_human_df_bot_counts_10plus.sort_values(ascending=False)

# Display the first 50 suspects
display(pr_comments_from_repos_with_human_df_bot_counts_10plus.head(50))

# Find Bots and change to BotOther
pr_comments_from_repos_with_human_bots_users =  ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot','facebook-github-bot']

pr_comments_from_repos_with_human_df.loc[pr_comments_from_repos_with_human_df['user'].isin(pr_comments_from_repos_with_human_bots_users), 'user_type'] = 'BotOther'

user
captainsafia       52
jakebailey         48
ian-at-airbyte     34
NoritakaIkeda      32
MayaRainer         27
vdaas-ci           25
wolfSSL-Bot        17
jairmyree          15
hairyhum           13
aishwaryabh        11
pytorchmergebot    11
Name: count, dtype: int64

### human_pr_comments_from_repos_with_human_df

In [5]:
# Filter user_type=="User"
human_pr_comments_from_repos_with_human_df_user_filter = human_pr_comments_from_repos_with_human_df['user_type'] == "User"

# Filter potential bots by username
human_pr_comments_from_repos_with_human_df_bot_like_filter = human_pr_comments_from_repos_with_human_df['user'].str.contains(r'bot|ci|copilot|ai|automation', case=False, na=False)

# Combine filters
human_pr_comments_from_repos_with_human_df_suspected_bots = human_pr_comments_from_repos_with_human_df[human_pr_comments_from_repos_with_human_df_user_filter & human_pr_comments_from_repos_with_human_df_bot_like_filter]

# Count the number of comments for each user
human_pr_comments_from_repos_with_human_df_bot_counts = human_pr_comments_from_repos_with_human_df_suspected_bots['user'].value_counts()

# Filter those who have a minimum of 10 comments
human_pr_comments_from_repos_with_human_df_bot_counts_10plus = human_pr_comments_from_repos_with_human_df_bot_counts[human_pr_comments_from_repos_with_human_df_bot_counts >= 10]

# Sort records by comment count
human_pr_comments_from_repos_with_human_df_bot_counts_10plus_sorted = human_pr_comments_from_repos_with_human_df_bot_counts_10plus.sort_values(ascending=False)

# Display the first 50 suspects
display(human_pr_comments_from_repos_with_human_df_bot_counts_10plus.head(50))

# Find Bots and change to BotOther
human_pr_comments_from_repos_with_human_bots_users =  ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot','facebook-github-bot']

human_pr_comments_from_repos_with_human_df.loc[human_pr_comments_from_repos_with_human_df['user'].isin(human_pr_comments_from_repos_with_human_bots_users), 'user_type'] = 'BotOther'

user
MayaRainer         36
jakebailey         23
calcom-bot         23
vdaas-ci           22
pytorchmergebot    19
typescript-bot     17
mikomikotaishi     17
lightdash-bot      10
Blaizzy            10
Name: count, dtype: int64

## Review Type

In [6]:
import re

AGENT_PATTERN = r"(copilot|coderabbit|coderabbitai|bugbot|gemini|claude|devin|" \
                r"sourcery-ai|greptile|giselles-ai|giselle|pytorchbot|pytorchmergebot|sentry|zwave-js-bot|create-docs-giselles-ai|dotnet-policy-service)"

COMMAND_PATTERN = (
    r"(run|retry|recheck|rebase|review|summari[sz]e|start|continue|fix|resolve|"\
    r"merge|test|format|lint|sign|add|remove|update|rerun|check|verify|resume|"\
    r"generate|create|solve|clean|commit|pack|try|thanks|conflicts)"
)

AGENT_CMD_RE = re.compile(
    rf"""(?:^|\s)
    ([@\\\/]?{AGENT_PATTERN})
    [^.\n]*?
    \b{COMMAND_PATTERN}\b
    |
    \b{COMMAND_PATTERN}\b
    [^.\n]*?
    ([@\/\\]?{AGENT_PATTERN})
    """,
    re.IGNORECASE | re.VERBOSE,
)

STARTS_WITH_AGENT = re.compile(
    rf"""
    ^[@/ ]?{AGENT_PATTERN}
    """,
    re.IGNORECASE | re.VERBOSE,
)

OTHER_COMMANDS = re.compile(
    r"^[/#!@\[\\](azp|check-enforcer|easycla|run|rebase|build-deploy-preview|backport|autoformat|test|bump-version|ci|ba-|format-|poe|"\
    r"cherrypick|huppy|poetry-|ok-|queue-|ci-|fix-|engineer|deploypr|update-|format|checkenforcer|approve-|review|gogoeditdiff"\
    r"|microsoft-github-policy-service|intelci|gpu|spinwick)",
    re.IGNORECASE | re.VERBOSE,
)
OTHER_COMMANDS_EXPL = re.compile(
    r"^(<\!-- buildkite-pr-bot-trigger-comment -->|\*\*\[chatops:help\]\*\*|\n\[\!\[coverage status\]|pre-commit\.ci autofix)", re.IGNORECASE)

WORKFLOW = re.compile(rf"^\s*(@\w+[\/\w\-]*\s+)?("\
    # --- Status/Closure/Merge Commands ---
    rf"resolve merge conflict(s)?"\
    rf"|merge (?:with master|main)"\
    rf"|closing (?:as stale|out prs created during experiments with devin\.|, integrated the temp fix in https?:\/\/\S+)"\
    rf"|this pull request was created as a test and will now be closed\.\s*thank you!?"\
    rf"|proactively closing this pull request\.\s*reopen if you need to after review\.\s*for any questions ping jwilcox\."\
    rf"|this pull request has been marked as stale due to inactivity\.\s*please update it to keep it open\."\
    rf"|this pull request has been closed due to lack of response\."\
    rf"|force-merging, per conversation in slack\.\s*future iterations can resolve the test failures and\/or lack of robust integration test environment\."\
    rf"|superseded by (?:pr #\d+|https?:\/\/\S+)"\
    # --- Simple Bot/Action Commands ---
    rf"|lgtm!?"\
    rf"|ok"\
    rf"|done"\
    rf"|good to go\."\
    rf"|good bot"\
    rf"|thank you(?: for|!)"\
    rf"|thanks"\
    rf"|thx"\
    rf"|crap"\
    rf"|nope"\
    rf"|bruh"\
    rf"|\!sync"\
    rf"|recheck"\
    rf"|try (?:again|this)"\
    rf"|please (?:try|retry|f)"\
    rf"|please test your latest changes by seeing if you're able to build\s*src\/core\/src\/core\.csproj"\
    rf"|can you retry working on this pr\?"\
    rf"|wake the fuck up samurai, we have prs to merge"\
    # --- Fix/CI Commands ---
    rf"|fix lint (?:issue|failure(?: in ci)?)"\
    rf"|lint (?:failure|ing fails)"\
    rf"|fix merge issues"\
    rf"|fix failing (?:tests|spec tests)\."\
    rf"|run (?:build|yarn change\" prerelease, |yarn lint:fix\" |yarn format\"|poetry lock and push it up|docs-build)"\
    rf"|make sure that the rust formatter has been run against all changed files, as this is currently causing the build to fail\."\
    rf"|i've fix with \[\w+\]\(https?:\/\/\S+\)"\
    rf"|revert all changes in the archive directory"\
    rf"|please remove the files under \.\/vendor\/\*"\
    rf"|a bunch of files have been added due to some whitespaces, clean out them and only include the files that have changes relevant to this issue\."\
    rf"|i think something went wrong in your last session, it seems like your changes weren't committed\. let's try again: please verify all of your changes and that the code works by running the json tests in stable and experimental mode"\
    rf"|it looks like there are still compile errors\.\s*can you try again\?"\
    rf"|please format the code and commit it back to the pr"\
    # --- Bot/System Status & Generic Error Messages ---
    rf"|build is still failing"\
    rf"|test are failing"\
    rf"|not working"\
    rf"|doesn't work"\
    rf"|the snippets failure is related to a preview feature\."\
    rf"|looks like your session timed out\. try again\."\
    rf"|already solved"\
    rf"|not a good solution"\
    rf"|the agent was blocked by configuration issues from accessing the necessary dependencies to successfully build and test\.\s*those are being fixed and we'll continue experimenting\."\
    rf"|unbelievable\. you, \[subject name here\], must be the pride of \[subject hometown here\]!"\
    rf"|it looks like the rebase was not correct\.\s*please start over with a fresh checkout from master, and make sure the build actually finishes \(\w*\) and that the new test actually runs and passes\.\s*even if the code looks simple the pr will not be accepted if the test doesn't actually run\."\
    rf"|can one of the admins verify this patch?"\
    # --- Action Requests on Documentation/Metadata ---
    rf"|add a bullet point to releasenotes\.md describing the change to our users"\
    rf"|update change log"\
    rf"|the description of the pr is not accurate anymore after all the changes, use examples from the new assert\.areequal tests and improve it to show some nice examples to the user\."\
    rf"|make sure to follow the github pr title structure like we are doing for integrations"\
    # --- Specific File Reversions ---
    rf"|revert all changes made to \"publicapi\.unshipped\.txt\" and then reapply changes to those files based on the api changes made in this pr"\
    rf"|the publicapi\.unshipped\.txt files need to be fixed\s*in order to fix them can you\s*1\) revert all the changes to publicapi\.unshipped\.txt\s*2\) and then make whatever additions you need to make to them in order to fix the build"\
    # --- User/Team Pings and Addressing Issues ---
    rf"|@\w+[-\/\w]*(?: \([\w\s:]+\))?(?: ping)?"\
    rf"|address #\d+"\
    rf"|related to https?:\/\/\S+"\
    rf"|i have read the cla document and i hereby sign the cla"\
    rf"|duplicate"\
    rf"|konfurikuto wo kaishou shite"\
    rf"|looks good to me"\
    # --- Closing the pattern ---
    rf")\s*",
    re.IGNORECASE | re.DOTALL)

AUTO_BOT = re.compile(
    rf"""
    ^\[approvalnotifier\]
    |^\s*hi\s+[@/]?{AGENT_PATTERN}\.\s+thanks\s+for\s+your\s+pr\..*
    |^\s*\*\*?\[WARNING:CONFLICT\]\*\*?.*
    |^\s*\*\*?\[FORMAT\]\*\*?.*
    |^\s*\*\*?API change check\*\*?.*
    |\:?tada\:
    """,
    re.IGNORECASE | re.VERBOSE,
)

def _matches(pattern, text: str) -> bool:
    return isinstance(text, str) and bool(pattern.search(text))


def is_agentic_ai_command(text: str) -> bool:
    return _matches(AGENT_CMD_RE, text) or _matches(STARTS_WITH_AGENT, text)

def is_other_command(text: str) -> bool:
    return _matches(OTHER_COMMANDS, text)

def is_auto_bot(text: str) -> bool:
    return _matches(AUTO_BOT, text) or _matches(OTHER_COMMANDS_EXPL, text)

def is_workflow(text: str) -> bool:
    return _matches(WORKFLOW, text)


def label_rev_type(text: str) -> str:
    if is_agentic_ai_command(text):
        return "agentic_cmd"
    if is_other_command(text) or is_auto_bot(text):
        return "bot"
    if is_workflow(text):
        return "workflow"
    return "review"

In [7]:
m_comments = pr_comments_df["user_type"] == "User"
m_bot = pr_comments_df["user_type"].isin(["Bot", "BotOther"])

pr_comments_df = pr_comments_df.copy()
pr_comments_df.loc[m_comments,"rev_type"] = pr_comments_df.loc[m_comments, "body"].apply(label_rev_type)
pr_comments_df.loc[m_bot, "rev_type"] = (pr_comments_df.loc[m_bot, "body"].apply(label_rev_type))

In [8]:
# mask for "real" users
m_human = human_pr_comments_from_repos_with_human_df["user_type"] == "User"
m_all = pr_comments_from_repos_with_human_df["user_type"] == "User"

# mask for "artificial" users
m_bot_human = human_pr_comments_from_repos_with_human_df["user_type"].isin(["Bot", "BotOther"])
m_bot = pr_comments_from_repos_with_human_df["user_type"].isin(["Bot", "BotOther"])

# error fix
human_pr_comments_from_repos_with_human_df = human_pr_comments_from_repos_with_human_df.copy()
pr_comments_from_repos_with_human_df = pr_comments_from_repos_with_human_df.copy()

# apply classifier only on those rows
human_pr_comments_from_repos_with_human_df.loc[m_human,"rev_type"] = human_pr_comments_from_repos_with_human_df.loc[m_human, "body"].apply(label_rev_type)
human_pr_comments_from_repos_with_human_df.loc[m_bot_human,"rev_type"] = human_pr_comments_from_repos_with_human_df.loc[m_bot_human, "body"].apply(label_rev_type)
pr_comments_from_repos_with_human_df.loc[m_all,"rev_type"] = pr_comments_from_repos_with_human_df.loc[m_all, "body"].apply(label_rev_type)
pr_comments_from_repos_with_human_df.loc[m_bot, "rev_type"] = (pr_comments_from_repos_with_human_df.loc[m_bot, "body"].apply(label_rev_type))

## Comments statistics for table

In [9]:
ai_comments_from_repos_with_human_agentic_comments_count = sum(pr_comments_from_repos_with_human_df["user_type"] != "User")
print(ai_comments_from_repos_with_human_agentic_comments_count)
human_comments_from_repos_with_human_agentic_comments_count = sum(human_pr_comments_from_repos_with_human_df["user_type"] != "User")
print(human_comments_from_repos_with_human_agentic_comments_count)

16727
5589


In [10]:
ai_comments_from_repos_with_human_human_comments_count = sum(pr_comments_from_repos_with_human_df["user_type"] == "User")
print(ai_comments_from_repos_with_human_human_comments_count)
human_comments_from_repos_with_human_human_comments_count = sum(human_pr_comments_from_repos_with_human_df["user_type"] == "User")
print(human_comments_from_repos_with_human_human_comments_count)

6354
4412


In [11]:
pr_comments_from_repos_with_human_df[pr_comments_from_repos_with_human_df['rev_type'] == 'bot']

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
424,2875765090,3057825906,lindy65,13020833,User,2025-05-13T09:38:34Z,/spinwick create --env MM_FEATUREFLAGS_Experim...,bot
1117,3018988146,3187477285,jfversluis,939291,User,2025-06-30T12:35:48Z,/azp run MAUI-public,bot
1143,2894278025,3076083247,T-Gro,46543583,User,2025-05-20T12:47:00Z,/run fantomas,bot
1149,2901448984,3076083247,T-Gro,46543583,User,2025-05-22T14:29:00Z,/run fantomas,bot
1151,2901463015,3076083247,T-Gro,46543583,User,2025-05-22T14:32:47Z,/run fantomas,bot
...,...,...,...,...,...,...,...,...
38814,2877718073,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:33:16Z,/bump-version type=minor,bot
38822,2877736008,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:37:23Z,/bump-version,bot
38825,2877745636,3060998787,theyueli,61070669,User,2025-05-13T19:40:27Z,"/bump-version type=patch changelog=""Add suppor...",bot
38827,2877751802,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:42:48Z,"/bump-version type=patch changelog=""Add suppor...",bot


In [12]:
pr_comments_from_repos_with_human_df__human = pr_comments_from_repos_with_human_df[m_all]
pr_comments_from_repos_with_human_df__human__counts = pr_comments_from_repos_with_human_df__human['rev_type'].value_counts()

In [13]:
human_pr_comments_from_repos_with_human_df__human = human_pr_comments_from_repos_with_human_df[m_human]
human_pr_comments_from_repos_with_human_df__human__counts = human_pr_comments_from_repos_with_human_df__human['rev_type'].value_counts()

## Silent PRs

### all PRs

In [14]:
# Loading all PRs from the full dataset
all_pr_df = pd.read_parquet("Data/pull_request.parquet")

# List all
all_pr_ids = all_pr_df["id"].unique()

# PRs with comments
prs_with_comments = pr_comments_df["pr_id"].unique()

# PRs without comments
prs_no_comments = set(all_pr_ids) - set(prs_with_comments)

print(f"All unique PRs: {len(all_pr_ids)}")
print(f"Silent PRs: {len(prs_no_comments)}")
print(f"PRs with any comment: {len(all_pr_ids) - len(prs_no_comments)}")

All unique PRs: 33596
Silent PRs: 20621
PRs with any comment: 12975


### ai PRs

In [15]:
all_ai_prs = all_prs_from_repos_with_human_prs["id"].unique()

# AI PRs with comments
ai_prs_with_comments = pr_comments_from_repos_with_human_df["pr_id"].unique()

# AI PRs without comments
ai_prs_no_comments = set(all_ai_prs) - set(ai_prs_with_comments)

ai_prs_no_comments_count = len(ai_prs_no_comments)

print(f"All unique PRs: {len(all_ai_prs)}")
print(f"Silent PRs: {len(ai_prs_no_comments)}")
print(f"PRs with any comment: {len(all_ai_prs) - len(ai_prs_no_comments)}")

All unique PRs: 9616
Silent PRs: 2781
PRs with any comment: 6835


### human PRs

In [16]:
all_human_prs = human_pr_df_without_bots["id"].unique()

# Human PRs with comments
human_prs_with_comments = human_pr_comments_from_repos_with_human_df["pr_id"].unique()

# human PRs without comments
human_prs_no_comments = set(all_human_prs) - set(human_prs_with_comments)

human_prs_no_comments_count = len(human_prs_no_comments)

print(f"All unique PRs: {len(all_human_prs)}")
print(f"Silent PRs: {len(human_prs_no_comments)}")
print(f"PRs with any comment: {len(all_human_prs) - len(human_prs_no_comments)}")

All unique PRs: 5574
Silent PRs: 1924
PRs with any comment: 3650


# Table no 1

### Functions

In [17]:
def compute_count_1(df):
    count_prs = df["pr_id"].nunique()

    count_reviewers_agentic = (
        df[
            df["user_type"].isin(["Bot", "BotOther"])
        ]["user"].nunique()
    )

    count_reviewers_human = (
        df[
            df["user_type"] == "User"
        ]["user"].nunique()
    )

    count_reviews_agentic = len(df[
        df["user_type"].isin(["Bot", "BotOther"])
    ])

    count_reviews_human = len(df[
        df["user_type"] == "User"
    ])

    print("Count # PRs: \t\t\t", count_prs)
    print("Count # Reviewers - agentic: \t", count_reviewers_agentic)
    print("Count # Reviewers - human: \t", count_reviewers_human)
    print("Count # Reviews - agentic: \t", count_reviews_human)
    print("Count # Reviews - human: \t", count_reviews_agentic)

    return count_prs, count_reviewers_agentic, count_reviewers_human, count_reviews_agentic, count_reviews_human, 

In [23]:
def compute_mean_std_1(df):
    all_reviews_per_pr = df.groupby("pr_id").size()
    all_mean = all_reviews_per_pr.mean()
    all_std  = all_reviews_per_pr.std()

    agentic_reviewers_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")["user"]
        .nunique()
    )
    agentic_rev_mean = agentic_reviewers_per_pr.mean()
    agentic_rev_std  = agentic_reviewers_per_pr.std()

    human_reviewers_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")["user"]
        .nunique()
    )
    human_rev_mean = human_reviewers_per_pr.mean()
    human_rev_std  = human_reviewers_per_pr.std()

    agentic_reviews_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")
        .size()
    )
    agentic_mean = agentic_reviews_per_pr.mean()
    agentic_std  = agentic_reviews_per_pr.std()

    human_reviews_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")
        .size()
    )
    human_mean = human_reviews_per_pr.mean()
    human_std  = human_reviews_per_pr.std()

    print("Mean + std PRs: \t\t", f"{all_mean:.2f} ± {all_std:.2f}")
    print("Mean + std Reviewers - agentic: ", f"{agentic_rev_mean:.2f} ± {agentic_rev_std:.2f}")
    print("Mean + std Reviewers - human: \t", f"{human_rev_mean:.2f} ± {human_rev_std:.2f}")
    print("Mean + std Reviews - agentic: \t", f"{agentic_mean:.2f} ± {agentic_std:.2f}")
    print("Mean + std Reviews - human: \t", f"{human_mean:.2f} ± {human_std:.2f}")

    return all_mean, all_std, agentic_rev_mean, agentic_rev_std, human_rev_mean, human_rev_std, agentic_mean, agentic_std, human_mean, human_std

In [21]:
import numpy as np

def median_ci(series, confidence=0.95, n_bootstrap=5000, random_state=42):
    random = np.random.default_rng(random_state)
    data = series.dropna().values
    n = len(data)

    boot_medians = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = random.choice(data, size=n, replace=True)
        boot_medians[i] = np.median(sample)

    alpha = (1 - confidence) / 2
    lower = np.quantile(boot_medians, alpha)
    upper = np.quantile(boot_medians, 1 - alpha)

    return np.median(data), lower, upper

def compute_median_ci_1(df):
    all_reviews_per_pr = df.groupby("pr_id").size()
    med_all, ci_all_low, ci_all_high = median_ci(all_reviews_per_pr)

    agentic_reviewers_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")["user"]
        .nunique()
    )
    med_agentic_reviewers, ci_ar_low, ci_ar_high = median_ci(agentic_reviewers_per_pr)

    human_reviewers_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")["user"]
        .nunique()
    )
    med_human_reviewers, ci_hr_low, ci_hr_high = median_ci(human_reviewers_per_pr)

    agentic_reviews_per_pr = (
        df[df["user_type"].isin(["Bot", "BotOther"])]
        .groupby("pr_id")
        .size()
    )
    med_agentic_reviews, ci_arev_low, ci_arev_high = median_ci(agentic_reviews_per_pr)


    human_reviews_per_pr = (
        df[df["user_type"] == "User"]
        .groupby("pr_id")
        .size()
    
    )
    med_human_reviews, ci_hrev_low, ci_hrev_high = median_ci(human_reviews_per_pr)

    print("Median + CI PRs: \t\t", f"{med_all:.2f} + [{ci_all_low:.2f} - {ci_all_high:.2f}]")
    print("Median + CI Reviewers - agentic: ", f"{med_agentic_reviewers:.2f} + [{ci_ar_low:.2f} - {ci_ar_high:.2f}]")
    print("Median + CI Reviewers - human: \t", f"{med_human_reviewers:.2f} + [{ci_hr_low:.2f} - {ci_hr_high:.2f}]")
    print("Median + CI Reviews - agentic: \t", f"{med_agentic_reviews:.2f} + [{ci_arev_low:.2f} - {ci_arev_high:.2f}]")
    print("Median + CI Reviews - human: \t", f"{med_human_reviews:.2f} + [{ci_hrev_low:.2f} - {ci_hrev_high:.2f}]")

    return med_all, ci_all_low, ci_all_high, med_agentic_reviewers, ci_ar_low, ci_ar_high, med_human_reviewers, ci_hr_low, ci_hr_high, \
        med_agentic_reviews, ci_arev_low, ci_arev_high, med_human_reviews, ci_hrev_low, ci_hrev_high

### pr_comments

In [19]:
t1_prc_count_prs, \
t1_prc_count_reviewers_agentic, t1_prc_count_reviewers_human, \
t1_prc_count_reviews_agentic, t1_prc_count_reviews_human = compute_count_1(pr_comments_df)

Count # PRs: 			 12975
Count # Reviewers - agentic: 	 144
Count # Reviewers - human: 	 2209
Count # Reviews - agentic: 	 11118
Count # Reviews - human: 	 28004


In [24]:
t1_prc_ms_prs_mean, t1_prc_ms_prs_std, \
t1_prc_ms_reviewers_agentic_mean, t1_prc_ms_reviewers_agentic_std, \
t1_prc_ms_reviewers_human_mean, t1_prc_ms_reviewers_human_std, \
t1_prc_ms_reviews_agentic_mean, t1_prc_ms_reviews_agentic_std, \
t1_prc_ms_reviews_human_mean, t1_prc_ms_reviews_human_std = compute_mean_std_1(pr_comments_df)

Mean + std PRs: 		 3.02 ± 3.16
Mean + std Reviewers - agentic:  1.69 ± 1.05
Mean + std Reviewers - human: 	 1.23 ± 0.66
Mean + std Reviews - agentic: 	 2.35 ± 1.95
Mean + std Reviews - human: 	 2.22 ± 2.42


In [25]:
t1_prc_me_ci_prs, t1_prc_me_ci_prs_low, t1_prc_me_ci_prs_high, \
t1_prc_me_ci_reviewers_agentic, t1_prc_me_ci_reviewers_agentic_low, t1_prc_me_ci_reviewers_agentic_high, \
t1_prc_me_ci_reviewers_human, t1_prc_me_ci_reviewers_human_low, t1_prc_me_ci_reviewers_human_high, \
t1_prc_me_ci_reviews_agentic, t1_prc_me_ci_reviews_agentic_low, t1_prc_me_ci_reviews_agentic_high, \
t1_prc_me_ci_reviews_human, t1_prc_me_ci_reviews_human_low, t1_prc_me_ci_reviews_human_high = compute_median_ci_1(pr_comments_df)

Median + CI PRs: 		 2.00 + [2.00 - 2.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 2.00 + [2.00 - 2.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


### pr_comments_from_repos_with_human_df

In [26]:
t1_prc_rwh_count_prs, \
t1_prc_rwh_count_reviewers_agentic, t1_prc_rwh_count_reviewers_human, \
t1_prc_rwh_count_reviews_agentic, t1_prc_rwh_count_reviews_human = compute_count_1(pr_comments_from_repos_with_human_df)

Count # PRs: 			 6835
Count # Reviewers - agentic: 	 84
Count # Reviewers - human: 	 1136
Count # Reviews - agentic: 	 6354
Count # Reviews - human: 	 16727


In [27]:
t1_prc_rwh_ms_prs_mean, t1_prc_rwh_ms_prs_std, \
t1_prc_rwh_ms_reviewers_agentic_mean, t1_prc_rwh_ms_reviewers_agentic_std, \
t1_prc_rwh_ms_reviewers_human_mean, t1_prc_rwh_ms_reviewers_human_std, \
t1_prc_rwh_ms_reviews_agentic_mean, t1_prc_rwh_ms_reviews_agentic_std, \
t1_prc_rwh_ms_reviews_human_mean, t1_prc_rwh_ms_rwh_reviews_human_std = compute_mean_std_1(pr_comments_from_repos_with_human_df)

Mean + std PRs: 		 3.38 ± 3.38
Mean + std Reviewers - agentic:  1.88 ± 1.21
Mean + std Reviewers - human: 	 1.24 ± 0.75
Mean + std Reviews - agentic: 	 2.66 ± 2.16
Mean + std Reviews - human: 	 2.19 ± 2.46


In [28]:
t1_prc_rwh_me_ci_prs, t1_prc_rwh_me_ci_prs_low, t1_prc_rwh_me_ci_prs_high, \
t1_prc_rwh_me_ci_reviewers_agentic, t1_prc_rwh_me_ci_reviewers_agentic_low, t1_prc_rwh_me_ci_reviewers_agentic_high, \
t1_prc_rwh_me_ci_reviewers_human, t1_prc_rwh_me_ci_reviewers_human_low, t1_prc_rwh_me_ci_reviewers_human_high, \
t1_prc_rwh_me_ci_reviews_agentic, t1_prc_rwh_me_ci_reviews_agentic_low, t1_prc_rwh_me_ci_reviews_agentic_high, \
t1_prc_rwh_me_ci_reviews_human, t1_prc_rwh_me_ci_reviews_human_low, t1_prc_rwh_me_ci_reviews_human_high = compute_median_ci_1(pr_comments_from_repos_with_human_df)

Median + CI PRs: 		 2.00 + [2.00 - 3.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 2.00 + [2.00 - 2.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


### human_pr_comments_from_repos_with_human_df

In [29]:
t1_hprc_rwh_count_prs, \
t1_hprc_rwh_count_reviewers_agentic, t1_hprc_rwh_count_reviewers_human, \
t1_hprc_rwh_count_reviews_agentic, t1_hprc_rwh_count_reviews_human = compute_count_1(human_pr_comments_from_repos_with_human_df)

Count # PRs: 			 3650
Count # Reviewers - agentic: 	 94
Count # Reviewers - human: 	 1401
Count # Reviews - agentic: 	 4412
Count # Reviews - human: 	 5589


In [30]:
t1_hprc_rwh_ms_prs_mean, t1_hprc_rwh_ms_prs_std, \
t1_hprc_rwh_ms_reviewers_agentic_mean, t1_hprc_rwh_ms_reviewers_agentic_std, \
t1_hprc_rwh_ms_reviewers_human_mean, t1_hprc_rwh_ms_reviewers_human_std, \
t1_hprc_rwh_ms_reviews_agentic_mean, t1_hprc_rwh_ms_reviews_agentic_std, \
t1_hprc_rwh_ms_reviews_human_mean, t1_hprc_rwh_ms_rwh_reviews_human_std = compute_mean_std_1(human_pr_comments_from_repos_with_human_df)

Mean + std PRs: 		 2.74 ± 2.99
Mean + std Reviewers - agentic:  1.68 ± 1.01
Mean + std Reviewers - human: 	 1.54 ± 0.85
Mean + std Reviews - agentic: 	 2.05 ± 1.66
Mean + std Reviews - human: 	 2.57 ± 3.32


In [31]:
t1_hprc_rwh_me_ci_prs, t1_hprc_rwh_me_ci_prs_low, t1_hprc_rwh_me_ci_prs_high, \
t1_hprc_rwh_me_ci_reviewers_agentic, t1_hprc_rwh_me_ci_reviewers_agentic_low, t1_hprc_rwh_me_ci_reviewers_agentic_high, \
t1_hprc_rwh_me_ci_reviewers_human, t1_hprc_rwh_me_ci_reviewers_human_low, t1_hprc_rwh_me_ci_reviewers_human_high, \
t1_hprc_rwh_me_ci_reviews_agentic, t1_hprc_rwh_me_ci_reviews_agentic_low, t1_hprc_rwh_me_ci_reviews_agentic_high, \
t1_hprc_rwh_me_ci_reviews_human, t1_hprc_rwh_me_ci_reviews_human_low, t1_hprc_rwh_me_ci_reviews_human_high = compute_median_ci_1(human_pr_comments_from_repos_with_human_df)

Median + CI PRs: 		 2.00 + [2.00 - 2.00]
Median + CI Reviewers - agentic:  1.00 + [1.00 - 1.00]
Median + CI Reviewers - human: 	 1.00 + [1.00 - 1.00]
Median + CI Reviews - agentic: 	 1.00 + [1.00 - 2.00]
Median + CI Reviews - human: 	 1.00 + [1.00 - 1.00]


### Table

In [37]:
index = pd.MultiIndex.from_tuples(
    [
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"),
        ("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"),
        ("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"),
        ("$C(P_{AI}| R)$", "\\# Reviewers", "Human"),
        ("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"),
        ("$C(P_{AI}| R)$", "\\# Reviews", "Human"),
        ("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"),
        ("$C(P_{human}| R)$", "\\# Reviewers", "Human"),
        ("$C(P_{human}| R)$", "\\# Reviews", "Agentic"),
        ("$C(P_{human}| R)$", "\\# Reviews", "Human"),
    ],
    names=["Dataset", "Category", "Type"]
)

columns = [
    "Count",
    "Mean ± Std",
    "Median, CI"
]

table1 = pd.DataFrame(np.nan, index=index, columns=columns)
table1["Count"] = table1["Count"].astype("Int64")

In [34]:
!pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]━━━━ 1/2 [jinja2]


In [38]:
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Count"] = t1_prc_count_reviewers_agentic
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Count"] = t1_prc_count_reviewers_human
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Count"] = t1_prc_count_reviews_agentic
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Count"] = t1_prc_count_reviews_human

table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_prc_ms_reviewers_agentic_mean:.2f} ± {t1_prc_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_prc_ms_reviewers_human_mean:.2f} ± {t1_prc_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_prc_ms_reviews_agentic_mean:.2f} ± {t1_prc_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_prc_ms_reviews_human_mean:.2f} ± {t1_prc_ms_reviews_human_std:.2f}"

table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_prc_me_ci_reviewers_agentic:.0f} + [{t1_prc_me_ci_reviewers_agentic_low:.0f} - {t1_prc_me_ci_reviewers_agentic_high:.0f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_prc_me_ci_reviewers_human:.0f} + [{t1_prc_me_ci_reviewers_human_low:.0f} - {t1_prc_me_ci_reviewers_human_high:.0f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_prc_me_ci_reviews_agentic:.0f} + [{t1_prc_me_ci_reviews_agentic_low:.0f} - {t1_prc_me_ci_reviews_agentic_high:.0f}]"
table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_prc_me_ci_reviews_human:.0f} + [{t1_prc_me_ci_reviews_human_low:.0f} - {t1_prc_me_ci_reviews_human_high:.0f}]"

table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Count"] = t1_prc_rwh_count_reviewers_agentic
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Count"] = t1_prc_rwh_count_reviewers_human
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Count"] = t1_prc_rwh_count_reviews_agentic
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Count"] = t1_prc_rwh_count_reviews_human

table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviewers_agentic_mean:.2f} ± {t1_prc_rwh_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviewers_human_mean:.2f} ± {t1_prc_rwh_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviews_agentic_mean:.2f} ± {t1_prc_rwh_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_prc_rwh_ms_reviews_human_mean:.2f} ± {t1_prc_rwh_ms_rwh_reviews_human_std:.2f}"

table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviewers_agentic:.0f} + [{t1_prc_rwh_me_ci_reviewers_agentic_low:.0f} - {t1_prc_rwh_me_ci_reviewers_agentic_high:.0f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviewers_human:.0f} + [{t1_prc_rwh_me_ci_reviewers_human_low:.0f} - {t1_prc_rwh_me_ci_reviewers_human_high:.0f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviews_agentic:.0f} + [{t1_prc_rwh_me_ci_reviews_agentic_low:.0f} - {t1_prc_rwh_me_ci_reviews_agentic_high:.0f}]"
table1.loc[("$C(P_{AI}| R)$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_prc_rwh_me_ci_reviews_human:.0f} + [{t1_prc_rwh_me_ci_reviews_human_low:.0f} - {t1_prc_rwh_me_ci_reviews_human_high:.0f}]"

table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Count"] = t1_hprc_rwh_count_reviewers_agentic
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Count"] = t1_hprc_rwh_count_reviewers_human
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Count"] = t1_hprc_rwh_count_reviews_agentic
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Count"] = t1_hprc_rwh_count_reviews_human

table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviewers_agentic_mean:.2f} ± {t1_hprc_rwh_ms_reviewers_agentic_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviewers_human_mean:.2f} ± {t1_hprc_rwh_ms_reviewers_human_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviews_agentic_mean:.2f} ± {t1_hprc_rwh_ms_reviews_agentic_std:.2f}"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Mean ± Std"] = f"{t1_hprc_rwh_ms_reviews_human_mean:.2f} ± {t1_hprc_rwh_ms_rwh_reviews_human_std:.2f}"

table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviewers_agentic:.0f} + [{t1_hprc_rwh_me_ci_reviewers_agentic_low:.0f} - {t1_hprc_rwh_me_ci_reviewers_agentic_high:.0f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviewers", "Human"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviewers_human:.0f} + [{t1_hprc_rwh_me_ci_reviewers_human_low:.0f} - {t1_hprc_rwh_me_ci_reviewers_human_high:.0f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Agentic"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviews_agentic:.0f} + [{t1_hprc_rwh_me_ci_reviews_agentic_low:.0f} - {t1_hprc_rwh_me_ci_reviews_agentic_high:.0f}]"
table1.loc[("$C(P_{human}| R)$", "\\# Reviews", "Human"), "Median, CI"] = f"{t1_hprc_rwh_me_ci_reviews_human:.0f} + [{t1_hprc_rwh_me_ci_reviews_human_low:.0f} - {t1_hprc_rwh_me_ci_reviews_human_high:.0f}]"

table1

/tmp/ipykernel_366649/3756425851.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.69 ± 1.05' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Mean ± Std"] = f"{t1_prc_ms_reviewers_agentic_mean:.2f} ± {t1_prc_ms_reviewers_agentic_std:.2f}"
/tmp/ipykernel_366649/3756425851.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1 + [1 - 1]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table1.loc[("$C(P_{AI}\\mid R_{pop})$", "\\# Reviewers", "Agentic"), "Median, CI"] = f"{t1_prc_me_ci_reviewers_agentic:.0f} + [{t1_prc_me_ci_reviewers_agentic_low:.0f} - {t1_prc_me_ci_reviewers_agentic_high:.0f}]"


Count   Mean ± Std   Median, CI
Dataset                 Category     Type                                    
$C(P_{AI}\mid R_{pop})$ \# Reviewers Agentic    144  1.69 ± 1.05  1 + [1 - 1]
                                     Human     2209  1.23 ± 0.66  1 + [1 - 1]
                        \# Reviews   Agentic  28004  2.35 ± 1.95  2 + [2 - 2]
                                     Human    11118  2.22 ± 2.42  1 + [1 - 1]
$C(P_{AI}| R)$          \# Reviewers Agentic     84  1.88 ± 1.21  1 + [1 - 1]
                                     Human     1136  1.24 ± 0.75  1 + [1 - 1]
                        \# Reviews   Agentic  16727  2.66 ± 2.16  2 + [2 - 2]
                                     Human     6354  2.19 ± 2.46  1 + [1 - 1]
$C(P_{human}| R)$       \# Reviewers Agentic     94  1.68 ± 1.01  1 + [1 - 1]
                                     Human     1401  1.54 ± 0.85  1 + [1 - 1]
                        \# Reviews   Agentic   5589  2.05 ± 1.66  1 + [1 - 2]
                                     Human     4412  2.57 ± 3.32  1 + [1 - 1]

In [39]:
#table1_reset = table1.reset_index()
latex = table1.to_latex(
    multirow=True,
    escape=False,
    index_names=True,
    header=True,       
)
latex = latex.replace("# PRs & -", r"\multicolumn{2}{l}{\# PRs}")
print(latex)

\begin{tabular}{lllrll}
\toprule
 &  &  & Count & Mean ± Std & Median, CI \\
Dataset & Category & Type &  &  &  \\
\midrule
\multirow[t]{4}{*}{$C(P_{AI}\mid R_{pop})$} & \multirow[t]{2}{*}{\# Reviewers} & Agentic & 144 & 1.69 ± 1.05 & 1 + [1 - 1] \\
 &  & Human & 2209 & 1.23 ± 0.66 & 1 + [1 - 1] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviews} & Agentic & 28004 & 2.35 ± 1.95 & 2 + [2 - 2] \\
 &  & Human & 11118 & 2.22 ± 2.42 & 1 + [1 - 1] \\
\cline{1-6} \cline{2-6}
\multirow[t]{4}{*}{$C(P_{AI}| R)$} & \multirow[t]{2}{*}{\# Reviewers} & Agentic & 84 & 1.88 ± 1.21 & 1 + [1 - 1] \\
 &  & Human & 1136 & 1.24 ± 0.75 & 1 + [1 - 1] \\
\cline{2-6}
 & \multirow[t]{2}{*}{\# Reviews} & Agentic & 16727 & 2.66 ± 2.16 & 2 + [2 - 2] \\
 &  & Human & 6354 & 2.19 ± 2.46 & 1 + [1 - 1] \\
\cline{1-6} \cline{2-6}
\multirow[t]{4}{*}{$C(P_{human}| R)$} & \multirow[t]{2}{*}{\# Reviewers} & Agentic & 94 & 1.68 ± 1.01 & 1 + [1 - 1] \\
 &  & Human & 1401 & 1.54 ± 0.85 & 1 + [1 - 1] \\
\cline{2-6}
 & \multirow[t

# Table no 2

### Functions

In [35]:
def calculate_2nd_table(df):
    rev_types = {
        "Evaluative": "review",
        "Agentic Cmd": "agentic_cmd",
        "Bot": "bot",
        "Workflow": "workflow",
    }

    results = {}

    for label, rt in rev_types.items():
        df_rt = df[df["rev_type"] == rt]

        count = len(df_rt)

        per_pr = df_rt.groupby("pr_id").size()

        mean = per_pr.mean()
        std = per_pr.std()

        median, ci_low, ci_high = median_ci(per_pr)

        results[label] = {
            "count": count,
            "mean": mean,
            "std": std,
            "median": median,
            "ci_low": ci_low,
            "ci_high": ci_high,
        }

    return results

### pr_comments

#### full

In [36]:
table_2nd_prc = calculate_2nd_table(pr_comments_df)

In [37]:
# Evaluative
eval_count_prc = table_2nd_prc["Evaluative"]["count"]
eval_mean_prc = table_2nd_prc["Evaluative"]["mean"]
eval_std_prc = table_2nd_prc["Evaluative"]["std"]
eval_median_prc = table_2nd_prc["Evaluative"]["median"]
eval_ci_low_prc = table_2nd_prc["Evaluative"]["ci_low"]
eval_ci_high_prc = table_2nd_prc["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc = table_2nd_prc["Agentic Cmd"]["count"]
cmd_mean_prc = table_2nd_prc["Agentic Cmd"]["mean"]
cmd_std_prc = table_2nd_prc["Agentic Cmd"]["std"]
cmd_median_prc = table_2nd_prc["Agentic Cmd"]["median"]
cmd_ci_low_prc = table_2nd_prc["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc = table_2nd_prc["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc = table_2nd_prc["Bot"]["count"]
bot_mean_prc = table_2nd_prc["Bot"]["mean"]
bot_std_prc = table_2nd_prc["Bot"]["std"]
bot_median_prc = table_2nd_prc["Bot"]["median"]
bot_ci_low_prc = table_2nd_prc["Bot"]["ci_low"]
bot_ci_high_prc = table_2nd_prc["Bot"]["ci_high"]

# Workflow
wf_count_prc = table_2nd_prc["Workflow"]["count"]
wf_mean_prc = table_2nd_prc["Workflow"]["mean"]
wf_std_prc = table_2nd_prc["Workflow"]["std"]
wf_median_prc = table_2nd_prc["Workflow"]["median"]
wf_ci_low_prc = table_2nd_prc["Workflow"]["ci_low"]
wf_ci_high_prc = table_2nd_prc["Workflow"]["ci_high"]


In [38]:
print("Evaluative")
print("Count: \t\t", eval_count_prc)
print("Mean + std: \t", f"{eval_mean_prc:.2f} ± {eval_std_prc:.2f}")
print("Median + CI: \t", f"{eval_median_prc:.2f} + [{eval_ci_low_prc:.2f} - {eval_ci_high_prc:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc)
print("Mean + std: \t", f"{cmd_mean_prc:.2f} ± {cmd_std_prc:.2f}")
print("Median + CI: \t", f"{cmd_median_prc:.2f} + [{cmd_ci_low_prc:.2f} - {cmd_ci_high_prc:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc)
print("Mean + std: \t", f"{bot_mean_prc:.2f} ± {bot_std_prc:.2f}")
print("Median + CI: \t", f"{bot_median_prc:.2f} + [{bot_ci_low_prc:.2f} - {bot_ci_high_prc:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc)
print("Mean + std: \t", f"{wf_mean_prc:.2f} ± {wf_std_prc:.2f}")
print("Median + CI: \t", f"{wf_median_prc:.2f} + [{wf_ci_low_prc:.2f} - {wf_ci_high_prc:.2f}]")

Evaluative
Count: 		 27964
Mean + std: 	 2.42 ± 2.19
Median + CI: 	 2.00 + [2.00 - 2.00]

Agentic Cmd
Count: 		 8412
Mean + std: 	 1.80 ± 1.95
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 950
Mean + std: 	 1.75 ± 1.73
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 1796
Mean + std: 	 1.44 ± 1.36
Median + CI: 	 1.00 + [1.00 - 1.00]


#### human

In [39]:
pr_comments_human = pr_comments_df[
    pr_comments_df["user_type"] == "User"
]

table_2nd_prc_human = calculate_2nd_table(pr_comments_human)

In [40]:
# Evaluative
eval_count_prc_human = table_2nd_prc_human["Evaluative"]["count"]
eval_mean_prc_human = table_2nd_prc_human["Evaluative"]["mean"]
eval_std_prc_human = table_2nd_prc_human["Evaluative"]["std"]
eval_median_prc_human = table_2nd_prc_human["Evaluative"]["median"]
eval_ci_low_prc_human = table_2nd_prc_human["Evaluative"]["ci_low"]
eval_ci_high_prc_human = table_2nd_prc_human["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc_human = table_2nd_prc_human["Agentic Cmd"]["count"]
cmd_mean_prc_human = table_2nd_prc_human["Agentic Cmd"]["mean"]
cmd_std_prc_human = table_2nd_prc_human["Agentic Cmd"]["std"]
cmd_median_prc_human = table_2nd_prc_human["Agentic Cmd"]["median"]
cmd_ci_low_prc_human = table_2nd_prc_human["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc_human = table_2nd_prc_human["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc_human = table_2nd_prc_human["Bot"]["count"]
bot_mean_prc_human = table_2nd_prc_human["Bot"]["mean"]
bot_std_prc_human = table_2nd_prc_human["Bot"]["std"]
bot_median_prc_human = table_2nd_prc_human["Bot"]["median"]
bot_ci_low_prc_human = table_2nd_prc_human["Bot"]["ci_low"]
bot_ci_high_prc_human = table_2nd_prc_human["Bot"]["ci_high"]

# Workflow
wf_count_prc_human = table_2nd_prc_human["Workflow"]["count"]
wf_mean_prc_human = table_2nd_prc_human["Workflow"]["mean"]
wf_std_prc_human = table_2nd_prc_human["Workflow"]["std"]
wf_median_prc_human = table_2nd_prc_human["Workflow"]["median"]
wf_ci_low_prc_human = table_2nd_prc_human["Workflow"]["ci_low"]
wf_ci_high_prc_human = table_2nd_prc_human["Workflow"]["ci_high"]


In [41]:
print("Evaluative")
print("Count: \t\t", eval_count_prc_human)
print("Mean + std: \t", f"{eval_mean_prc_human:.2f} ± {eval_std_prc_human:.2f}")
print("Median + CI: \t", f"{eval_median_prc_human:.2f} + [{eval_ci_low_prc_human:.2f} - {eval_ci_high_prc_human:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc_human)
print("Mean + std: \t", f"{cmd_mean_prc_human:.2f} ± {cmd_std_prc_human:.2f}")
print("Median + CI: \t", f"{cmd_median_prc_human:.2f} + [{cmd_ci_low_prc_human:.2f} - {cmd_ci_high_prc_human:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc_human)
print("Mean + std: \t", f"{bot_mean_prc_human:.2f} ± {bot_std_prc_human:.2f}")
print("Median + CI: \t", f"{bot_median_prc_human:.2f} + [{bot_ci_low_prc_human:.2f} - {bot_ci_high_prc_human:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc_human)
print("Mean + std: \t", f"{wf_mean_prc_human:.2f} ± {wf_std_prc_human:.2f}")
print("Median + CI: \t", f"{wf_median_prc_human:.2f} + [{wf_ci_low_prc_human:.2f} - {wf_ci_high_prc_human:.2f}]")

Evaluative
Count: 		 5767
Mean + std: 	 1.73 ± 1.65
Median + CI: 	 1.00 + [1.00 - 1.00]

Agentic Cmd
Count: 		 3154
Mean + std: 	 2.01 ± 2.05
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 789
Mean + std: 	 1.72 ± 1.55
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 1408
Mean + std: 	 1.40 ± 1.05
Median + CI: 	 1.00 + [1.00 - 1.00]


#### agentic

In [42]:
pr_comments_ai = pr_comments_df[
    pr_comments_df["user_type"].isin(["Bot", "BotOther"])
]

table_2nd_prc_agentic = calculate_2nd_table(pr_comments_ai)

In [43]:
# Evaluative
eval_count_prc_agentic = table_2nd_prc_agentic["Evaluative"]["count"]
eval_mean_prc_agentic = table_2nd_prc_agentic["Evaluative"]["mean"]
eval_std_prc_agentic = table_2nd_prc_agentic["Evaluative"]["std"]
eval_median_prc_agentic = table_2nd_prc_agentic["Evaluative"]["median"]
eval_ci_low_prc_agentic = table_2nd_prc_agentic["Evaluative"]["ci_low"]
eval_ci_high_prc_agentic = table_2nd_prc_agentic["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["count"]
cmd_mean_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["mean"]
cmd_std_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["std"]
cmd_median_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["median"]
cmd_ci_low_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc_agentic = table_2nd_prc_agentic["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc_agentic = table_2nd_prc_agentic["Bot"]["count"]
bot_mean_prc_agentic = table_2nd_prc_agentic["Bot"]["mean"]
bot_std_prc_agentic = table_2nd_prc_agentic["Bot"]["std"]
bot_median_prc_agentic = table_2nd_prc_agentic["Bot"]["median"]
bot_ci_low_prc_agentic = table_2nd_prc_agentic["Bot"]["ci_low"]
bot_ci_high_prc_agentic = table_2nd_prc_agentic["Bot"]["ci_high"]

# Workflow
wf_count_prc_agentic = table_2nd_prc_agentic["Workflow"]["count"]
wf_mean_prc_agentic = table_2nd_prc_agentic["Workflow"]["mean"]
wf_std_prc_agentic = table_2nd_prc_agentic["Workflow"]["std"]
wf_median_prc_agentic = table_2nd_prc_agentic["Workflow"]["median"]
wf_ci_low_prc_agentic = table_2nd_prc_agentic["Workflow"]["ci_low"]
wf_ci_high_prc_agentic = table_2nd_prc_agentic["Workflow"]["ci_high"]

In [44]:
print("Evaluative")
print("Count: \t\t", eval_count_prc_agentic)
print("Mean + std: \t", f"{eval_mean_prc_agentic:.2f} ± {eval_std_prc_agentic:.2f}")
print("Median + CI: \t", f"{eval_median_prc_agentic:.2f} + [{eval_ci_low_prc_agentic:.2f} - {eval_ci_high_prc_agentic:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc_agentic)
print("Mean + std: \t", f"{cmd_mean_prc_agentic:.2f} ± {cmd_std_prc_agentic:.2f}")
print("Median + CI: \t", f"{cmd_median_prc_agentic:.2f} + [{cmd_ci_low_prc_agentic:.2f} - {cmd_ci_high_prc_agentic:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc_agentic)
print("Mean + std: \t", f"{bot_mean_prc_agentic:.2f} ± {bot_std_prc_agentic:.2f}")
print("Median + CI: \t", f"{bot_median_prc_agentic:.2f} + [{bot_ci_low_prc_agentic:.2f} - {bot_ci_high_prc_agentic:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc_agentic)
print("Mean + std: \t", f"{wf_mean_prc_agentic:.2f} ± {wf_std_prc_agentic:.2f}")
print("Median + CI: \t", f"{wf_median_prc_agentic:.2f} + [{wf_ci_low_prc_agentic:.2f} - {wf_ci_high_prc_agentic:.2f}]")

Evaluative
Count: 		 22197
Mean + std: 	 2.09 ± 1.66
Median + CI: 	 2.00 + [2.00 - 2.00]

Agentic Cmd
Count: 		 5258
Mean + std: 	 1.31 ± 0.78
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 161
Mean + std: 	 1.66 ± 2.17
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 388
Mean + std: 	 1.44 ± 2.04
Median + CI: 	 1.00 + [1.00 - 1.00]


### pr_comments_from_repos_with_human_df

#### full

In [45]:
table_2nd_prc_rph = calculate_2nd_table(pr_comments_from_repos_with_human_df)

In [46]:
# Evaluative
eval_count_prc_rph = table_2nd_prc_rph["Evaluative"]["count"]
eval_mean_prc_rph = table_2nd_prc_rph["Evaluative"]["mean"]
eval_std_prc_rph = table_2nd_prc_rph["Evaluative"]["std"]
eval_median_prc_rph = table_2nd_prc_rph["Evaluative"]["median"]
eval_ci_low_prc_rph = table_2nd_prc_rph["Evaluative"]["ci_low"]
eval_ci_high_prc_rph = table_2nd_prc_rph["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["count"]
cmd_mean_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["mean"]
cmd_std_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["std"]
cmd_median_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["median"]
cmd_ci_low_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc_rph = table_2nd_prc_rph["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc_rph = table_2nd_prc_rph["Bot"]["count"]
bot_mean_prc_rph = table_2nd_prc_rph["Bot"]["mean"]
bot_std_prc_rph = table_2nd_prc_rph["Bot"]["std"]
bot_median_prc_rph = table_2nd_prc_rph["Bot"]["median"]
bot_ci_low_prc_rph = table_2nd_prc_rph["Bot"]["ci_low"]
bot_ci_high_prc_rph = table_2nd_prc_rph["Bot"]["ci_high"]

# Workflow
wf_count_prc_rph = table_2nd_prc_rph["Workflow"]["count"]
wf_mean_prc_rph = table_2nd_prc_rph["Workflow"]["mean"]
wf_std_prc_rph = table_2nd_prc_rph["Workflow"]["std"]
wf_median_prc_rph = table_2nd_prc_rph["Workflow"]["median"]
wf_ci_low_prc_rph = table_2nd_prc_rph["Workflow"]["ci_low"]
wf_ci_high_prc_rph = table_2nd_prc_rph["Workflow"]["ci_high"]


In [47]:
print("Evaluative")
print("Count: \t\t", eval_count_prc_rph)
print("Mean + std: \t", f"{eval_mean_prc_rph:.2f} ± {eval_std_prc_rph:.2f}")
print("Median + CI: \t", f"{eval_median_prc_rph:.2f} + [{eval_ci_low_prc_rph:.2f} - {eval_ci_high_prc_rph:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc_rph)
print("Mean + std: \t", f"{cmd_mean_prc_rph:.2f} ± {cmd_std_prc_rph:.2f}")
print("Median + CI: \t", f"{cmd_median_prc_rph:.2f} + [{cmd_ci_low_prc_rph:.2f} - {cmd_ci_high_prc_rph:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc_rph)
print("Mean + std: \t", f"{bot_mean_prc_rph:.2f} ± {bot_std_prc_rph:.2f}")
print("Median + CI: \t", f"{bot_median_prc_rph:.2f} + [{bot_ci_low_prc_rph:.2f} - {bot_ci_high_prc_rph:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc_rph)
print("Mean + std: \t", f"{wf_mean_prc_rph:.2f} ± {wf_std_prc_rph:.2f}")
print("Median + CI: \t", f"{wf_median_prc_rph:.2f} + [{wf_ci_low_prc_rph:.2f} - {wf_ci_high_prc_rph:.2f}]")

Evaluative
Count: 		 16914
Mean + std: 	 2.64 ± 2.36
Median + CI: 	 2.00 + [2.00 - 2.00]

Agentic Cmd
Count: 		 4463
Mean + std: 	 1.81 ± 1.89
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 653
Mean + std: 	 1.87 ± 1.96
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 1051
Mean + std: 	 1.49 ± 1.61
Median + CI: 	 1.00 + [1.00 - 1.00]


#### human

In [48]:
pr_comments_rwh_human = pr_comments_from_repos_with_human_df[
    pr_comments_from_repos_with_human_df["user_type"] == "User"
]

table_2nd_prc_rph_human = calculate_2nd_table(pr_comments_rwh_human)

In [49]:
# Evaluative
eval_count_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["count"]
eval_mean_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["mean"]
eval_std_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["std"]
eval_median_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["median"]
eval_ci_low_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["ci_low"]
eval_ci_high_prc_rph_human = table_2nd_prc_rph_human["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["count"]
cmd_mean_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["mean"]
cmd_std_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["std"]
cmd_median_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["median"]
cmd_ci_low_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc_rph_human = table_2nd_prc_rph_human["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc_rph_human = table_2nd_prc_rph_human["Bot"]["count"]
bot_mean_prc_rph_human = table_2nd_prc_rph_human["Bot"]["mean"]
bot_std_prc_rph_human = table_2nd_prc_rph_human["Bot"]["std"]
bot_median_prc_rph_human = table_2nd_prc_rph_human["Bot"]["median"]
bot_ci_low_prc_rph_human = table_2nd_prc_rph_human["Bot"]["ci_low"]
bot_ci_high_prc_rph_human = table_2nd_prc_rph_human["Bot"]["ci_high"]

# Workflow
wf_count_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["count"]
wf_mean_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["mean"]
wf_std_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["std"]
wf_median_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["median"]
wf_ci_low_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["ci_low"]
wf_ci_high_prc_rph_human = table_2nd_prc_rph_human["Workflow"]["ci_high"]


In [50]:
print("Evaluative")
print("Count: \t\t", eval_count_prc_rph_human)
print("Mean + std: \t", f"{eval_mean_prc_rph_human:.2f} ± {eval_std_prc_rph_human:.2f}")
print("Median + CI: \t", f"{eval_median_prc_rph_human:.2f} + [{eval_ci_low_prc_rph_human:.2f} - {eval_ci_high_prc_rph_human:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc_rph_human)
print("Mean + std: \t", f"{cmd_mean_prc_rph_human:.2f} ± {cmd_std_prc_rph_human:.2f}")
print("Median + CI: \t", f"{cmd_median_prc_rph_human:.2f} + [{cmd_ci_low_prc_rph_human:.2f} - {cmd_ci_high_prc_rph_human:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc_rph_human)
print("Mean + std: \t", f"{bot_mean_prc_rph_human:.2f} ± {bot_std_prc_rph_human:.2f}")
print("Median + CI: \t", f"{bot_median_prc_rph_human:.2f} + [{bot_ci_low_prc_rph_human:.2f} - {bot_ci_high_prc_rph_human:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc_rph_human)
print("Mean + std: \t", f"{wf_mean_prc_rph_human:.2f} ± {wf_std_prc_rph_human:.2f}")
print("Median + CI: \t", f"{wf_median_prc_rph_human:.2f} + [{wf_ci_low_prc_rph_human:.2f} - {wf_ci_high_prc_rph_human:.2f}]")

Evaluative
Count: 		 3333
Mean + std: 	 1.70 ± 1.69
Median + CI: 	 1.00 + [1.00 - 1.00]

Agentic Cmd
Count: 		 1661
Mean + std: 	 1.94 ± 1.79
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 568
Mean + std: 	 1.80 ± 1.66
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 792
Mean + std: 	 1.40 ± 1.14
Median + CI: 	 1.00 + [1.00 - 1.00]


#### agentic

In [51]:
pr_comments_rwh_ai = pr_comments_from_repos_with_human_df[
    pr_comments_from_repos_with_human_df["user_type"].isin(["Bot", "BotOther"])
]

table_2nd_prc_rph_agentic = calculate_2nd_table(pr_comments_rwh_ai)

In [52]:
# Evaluative
eval_count_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["count"]
eval_mean_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["mean"]
eval_std_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["std"]
eval_median_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["median"]
eval_ci_low_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["ci_low"]
eval_ci_high_prc_rph_agentic = table_2nd_prc_rph_agentic["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["count"]
cmd_mean_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["mean"]
cmd_std_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["std"]
cmd_median_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["median"]
cmd_ci_low_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["ci_low"]
cmd_ci_high_prc_rph_agentic = table_2nd_prc_rph_agentic["Agentic Cmd"]["ci_high"]

# Bot
bot_count_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["count"]
bot_mean_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["mean"]
bot_std_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["std"]
bot_median_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["median"]
bot_ci_low_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["ci_low"]
bot_ci_high_prc_rph_agentic = table_2nd_prc_rph_agentic["Bot"]["ci_high"]

# Workflow
wf_count_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["count"]
wf_mean_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["mean"]
wf_std_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["std"]
wf_median_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["median"]
wf_ci_low_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["ci_low"]
wf_ci_high_prc_rph_agentic = table_2nd_prc_rph_agentic["Workflow"]["ci_high"]


In [53]:
print("Evaluative")
print("Count: \t\t", eval_count_prc_rph_agentic)
print("Mean + std: \t", f"{eval_mean_prc_rph_agentic:.2f} ± {eval_std_prc_rph_agentic:.2f}")
print("Median + CI: \t", f"{eval_median_prc_rph_agentic:.2f} + [{eval_ci_low_prc_rph_agentic:.2f} - {eval_ci_high_prc_rph_agentic:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_prc_rph_agentic)
print("Mean + std: \t", f"{cmd_mean_prc_rph_agentic:.2f} ± {cmd_std_prc_rph_agentic:.2f}")
print("Median + CI: \t", f"{cmd_median_prc_rph_agentic:.2f} + [{cmd_ci_low_prc_rph_agentic:.2f} - {cmd_ci_high_prc_rph_agentic:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_prc_rph_agentic)
print("Mean + std: \t", f"{bot_mean_prc_rph_agentic:.2f} ± {bot_std_prc_rph_agentic:.2f}")
print("Median + CI: \t", f"{bot_median_prc_rph_agentic:.2f} + [{bot_ci_low_prc_rph_agentic:.2f} - {bot_ci_high_prc_rph_agentic:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_prc_rph_agentic)
print("Mean + std: \t", f"{wf_mean_prc_rph_agentic:.2f} ± {wf_std_prc_rph_agentic:.2f}")
print("Median + CI: \t", f"{wf_median_prc_rph_agentic:.2f} + [{wf_ci_low_prc_rph_agentic:.2f} - {wf_ci_high_prc_rph_agentic:.2f}]")

Evaluative
Count: 		 13581
Mean + std: 	 2.29 ± 1.83
Median + CI: 	 2.00 + [2.00 - 2.00]

Agentic Cmd
Count: 		 2802
Mean + std: 	 1.34 ± 0.81
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 85
Mean + std: 	 1.89 ± 3.11
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 259
Mean + std: 	 1.60 ± 2.55
Median + CI: 	 1.00 + [1.00 - 1.00]


### human_pr_comments_from_repos_with_human_df

#### full

In [54]:
table_2nd_hprc_rph = calculate_2nd_table(human_pr_comments_from_repos_with_human_df)

In [55]:
# Evaluative
eval_count_hprc_rph = table_2nd_hprc_rph["Evaluative"]["count"]
eval_mean_hprc_rph = table_2nd_hprc_rph["Evaluative"]["mean"]
eval_std_hprc_rph = table_2nd_hprc_rph["Evaluative"]["std"]
eval_median_hprc_rph = table_2nd_hprc_rph["Evaluative"]["median"]
eval_ci_low_hprc_rph = table_2nd_hprc_rph["Evaluative"]["ci_low"]
eval_ci_high_hprc_rph = table_2nd_hprc_rph["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["count"]
cmd_mean_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["mean"]
cmd_std_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["std"]
cmd_median_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["median"]
cmd_ci_low_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["ci_low"]
cmd_ci_high_hprc_rph = table_2nd_hprc_rph["Agentic Cmd"]["ci_high"]

# Bot
bot_count_hprc_rph = table_2nd_hprc_rph["Bot"]["count"]
bot_mean_hprc_rph = table_2nd_hprc_rph["Bot"]["mean"]
bot_std_hprc_rph = table_2nd_hprc_rph["Bot"]["std"]
bot_median_hprc_rph = table_2nd_hprc_rph["Bot"]["median"]
bot_ci_low_hprc_rph = table_2nd_hprc_rph["Bot"]["ci_low"]
bot_ci_high_hprc_rph = table_2nd_hprc_rph["Bot"]["ci_high"]

# Workflow
wf_count_hprc_rph = table_2nd_hprc_rph["Workflow"]["count"]
wf_mean_hprc_rph = table_2nd_hprc_rph["Workflow"]["mean"]
wf_std_hprc_rph = table_2nd_hprc_rph["Workflow"]["std"]
wf_median_hprc_rph = table_2nd_hprc_rph["Workflow"]["median"]
wf_ci_low_hprc_rph = table_2nd_hprc_rph["Workflow"]["ci_low"]
wf_ci_high_hprc_rph = table_2nd_hprc_rph["Workflow"]["ci_high"]


In [56]:
print("Evaluative")
print("Count: \t\t", eval_count_hprc_rph)
print("Mean + std: \t", f"{eval_mean_hprc_rph:.2f} ± {eval_std_hprc_rph:.2f}")
print("Median + CI: \t", f"{eval_median_hprc_rph:.2f} + [{eval_ci_low_hprc_rph:.2f} - {eval_ci_high_hprc_rph:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_hprc_rph)
print("Mean + std: \t", f"{cmd_mean_hprc_rph:.2f} ± {cmd_std_hprc_rph:.2f}")
print("Median + CI: \t", f"{cmd_median_hprc_rph:.2f} + [{cmd_ci_low_hprc_rph:.2f} - {cmd_ci_high_hprc_rph:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_hprc_rph)
print("Mean + std: \t", f"{bot_mean_hprc_rph:.2f} ± {bot_std_hprc_rph:.2f}")
print("Median + CI: \t", f"{bot_median_hprc_rph:.2f} + [{bot_ci_low_hprc_rph:.2f} - {bot_ci_high_hprc_rph:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_hprc_rph)
print("Mean + std: \t", f"{wf_mean_hprc_rph:.2f} ± {wf_std_hprc_rph:.2f}")
print("Median + CI: \t", f"{wf_median_hprc_rph:.2f} + [{wf_ci_low_hprc_rph:.2f} - {wf_ci_high_hprc_rph:.2f}]")

Evaluative
Count: 		 7857
Mean + std: 	 2.39 ± 2.47
Median + CI: 	 2.00 + [2.00 - 2.00]

Agentic Cmd
Count: 		 635
Mean + std: 	 1.13 ± 0.63
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 269
Mean + std: 	 1.71 ± 1.87
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 1240
Mean + std: 	 1.65 ± 1.43
Median + CI: 	 1.00 + [1.00 - 1.00]


#### human

In [57]:
human_pr_comments_rwh_human = human_pr_comments_from_repos_with_human_df[
    human_pr_comments_from_repos_with_human_df["user_type"] == "User"
]

table_2nd_hprc_rph_human = calculate_2nd_table(human_pr_comments_rwh_human)

In [58]:
# Evaluative
eval_count_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["count"]
eval_mean_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["mean"]
eval_std_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["std"]
eval_median_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["median"]
eval_ci_low_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["ci_low"]
eval_ci_high_hprc_rph_human = table_2nd_hprc_rph_human["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["count"]
cmd_mean_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["mean"]
cmd_std_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["std"]
cmd_median_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["median"]
cmd_ci_low_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["ci_low"]
cmd_ci_high_hprc_rph_human = table_2nd_hprc_rph_human["Agentic Cmd"]["ci_high"]

# Bot
bot_count_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["count"]
bot_mean_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["mean"]
bot_std_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["std"]
bot_median_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["median"]
bot_ci_low_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["ci_low"]
bot_ci_high_hprc_rph_human = table_2nd_hprc_rph_human["Bot"]["ci_high"]

# Workflow
wf_count_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["count"]
wf_mean_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["mean"]
wf_std_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["std"]
wf_median_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["median"]
wf_ci_low_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["ci_low"]
wf_ci_high_hprc_rph_human = table_2nd_hprc_rph_human["Workflow"]["ci_high"]


In [59]:
print("Evaluative")
print("Count: \t\t", eval_count_hprc_rph_human)
print("Mean + std: \t", f"{eval_mean_hprc_rph_human:.2f} ± {eval_std_hprc_rph_human:.2f}")
print("Median + CI: \t", f"{eval_median_hprc_rph_human:.2f} + [{eval_ci_low_hprc_rph_human:.2f} - {eval_ci_high_hprc_rph_human:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_hprc_rph_human)
print("Mean + std: \t", f"{cmd_mean_hprc_rph_human:.2f} ± {cmd_std_hprc_rph_human:.2f}")
print("Median + CI: \t", f"{cmd_median_hprc_rph_human:.2f} + [{cmd_ci_low_hprc_rph_human:.2f} - {cmd_ci_high_hprc_rph_human:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_hprc_rph_human)
print("Mean + std: \t", f"{bot_mean_hprc_rph_human:.2f} ± {bot_std_hprc_rph_human:.2f}")
print("Median + CI: \t", f"{bot_median_hprc_rph_human:.2f} + [{bot_ci_low_hprc_rph_human:.2f} - {bot_ci_high_hprc_rph_human:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_hprc_rph_human)
print("Mean + std: \t", f"{wf_mean_hprc_rph_human:.2f} ± {wf_std_hprc_rph_human:.2f}")
print("Median + CI: \t", f"{wf_median_hprc_rph_human:.2f} + [{wf_ci_low_hprc_rph_human:.2f} - {wf_ci_high_hprc_rph_human:.2f}]")

Evaluative
Count: 		 3089
Mean + std: 	 2.20 ± 2.81
Median + CI: 	 1.00 + [1.00 - 1.00]

Agentic Cmd
Count: 		 80
Mean + std: 	 1.43 ± 1.11
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 223
Mean + std: 	 1.87 ± 2.09
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 1020
Mean + std: 	 1.66 ± 1.46
Median + CI: 	 1.00 + [1.00 - 1.00]


#### agentic

In [60]:
human_pr_comments_rwh_ai = human_pr_comments_from_repos_with_human_df[
    human_pr_comments_from_repos_with_human_df["user_type"].isin(["Bot", "BotOther"])
]

table_2nd_hprc_rph_agentic = calculate_2nd_table(human_pr_comments_rwh_ai)

In [61]:
# Evaluative
eval_count_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["count"]
eval_mean_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["mean"]
eval_std_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["std"]
eval_median_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["median"]
eval_ci_low_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["ci_low"]
eval_ci_high_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Evaluative"]["ci_high"]

# Agentic Cmd
cmd_count_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["count"]
cmd_mean_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["mean"]
cmd_std_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["std"]
cmd_median_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["median"]
cmd_ci_low_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["ci_low"]
cmd_ci_high_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Agentic Cmd"]["ci_high"]

# Bot
bot_count_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["count"]
bot_mean_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["mean"]
bot_std_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["std"]
bot_median_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["median"]
bot_ci_low_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["ci_low"]
bot_ci_high_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Bot"]["ci_high"]

# Workflow
wf_count_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["count"]
wf_mean_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["mean"]
wf_std_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["std"]
wf_median_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["median"]
wf_ci_low_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["ci_low"]
wf_ci_high_hprc_rph_agentic = table_2nd_hprc_rph_agentic["Workflow"]["ci_high"]


In [62]:
print("Evaluative")
print("Count: \t\t", eval_count_hprc_rph_agentic)
print("Mean + std: \t", f"{eval_mean_hprc_rph_agentic:.2f} ± {eval_std_hprc_rph_agentic:.2f}")
print("Median + CI: \t", f"{eval_median_hprc_rph_agentic:.2f} + [{eval_ci_low_hprc_rph_agentic:.2f} - {eval_ci_high_hprc_rph_agentic:.2f}]")

print("\nAgentic Cmd")
print("Count: \t\t", cmd_count_hprc_rph_agentic)
print("Mean + std: \t", f"{cmd_mean_hprc_rph_agentic:.2f} ± {cmd_std_hprc_rph_agentic:.2f}")
print("Median + CI: \t", f"{cmd_median_hprc_rph_agentic:.2f} + [{cmd_ci_low_hprc_rph_agentic:.2f} - {cmd_ci_high_hprc_rph_agentic:.2f}]")

print("\nBot")
print("Count: \t\t", bot_count_hprc_rph_agentic)
print("Mean + std: \t", f"{bot_mean_hprc_rph_agentic:.2f} ± {bot_std_hprc_rph_agentic:.2f}")
print("Median + CI: \t", f"{bot_median_hprc_rph_agentic:.2f} + [{bot_ci_low_hprc_rph_agentic:.2f} - {bot_ci_high_hprc_rph_agentic:.2f}]")

print("\nWorkflow")
print("Count: \t\t", wf_count_hprc_rph_agentic)
print("Mean + std: \t", f"{wf_mean_hprc_rph_agentic:.2f} ± {wf_std_hprc_rph_agentic:.2f}")
print("Median + CI: \t", f"{wf_median_hprc_rph_agentic:.2f} + [{wf_ci_low_hprc_rph_agentic:.2f} - {wf_ci_high_hprc_rph_agentic:.2f}]")

Evaluative
Count: 		 4768
Mean + std: 	 1.99 ± 1.66
Median + CI: 	 1.00 + [1.00 - 1.00]

Agentic Cmd
Count: 		 555
Mean + std: 	 1.04 ± 0.26
Median + CI: 	 1.00 + [1.00 - 1.00]

Bot
Count: 		 46
Mean + std: 	 1.18 ± 0.51
Median + CI: 	 1.00 + [1.00 - 1.00]

Workflow
Count: 		 220
Mean + std: 	 1.06 ± 0.40
Median + CI: 	 1.00 + [1.00 - 1.00]


### Table

In [63]:
columns = pd.MultiIndex.from_tuples([
    ("", "PR author"),
    ("", "Repository set"),
    ("Review status", "$C_0$"),
    ("Review status", "$C_{AI}$"),
    ("Review status", "$C_{human}$"),
    ("Human review categories", "Evaluative"),
    ("Human review categories", "Agentic"),
    ("Human review categories", "Bot"),
    ("Human review categories", "Workflow"),
])

pr_comments_df__human__counts = pd.Series({
    "review": 5767,
    "agentic_cmd": 3154,
    "workflow": 1408,
    "bot": 789
})

pr_comments_no_comments_count = 20621
pr_comments_human_comments_count = 11118
pr_comments_agentic_comments_count = 28004

ai_agent_r_pop_sum = pr_comments_no_comments_count + pr_comments_agentic_comments_count + pr_comments_human_comments_count
ai_agent_r_sum = ai_prs_no_comments_count + ai_comments_from_repos_with_human_agentic_comments_count + ai_comments_from_repos_with_human_human_comments_count
human_r_sum = human_prs_no_comments_count + human_comments_from_repos_with_human_agentic_comments_count + human_comments_from_repos_with_human_human_comments_count

def formattt(x, y):
    return f"{x} " + f"({round(x * 100 / y, 2)}\%)"


table2 = pd.DataFrame(columns=columns)
table2[("", "PR author")] = ["AI agent", "AI agent", "Human"]
table2[("", "Repository set")] = ["$R_{pop}$", "$R$", "$R$"]
table2[("Review status", "$C_0$")] = [
    formattt(pr_comments_no_comments_count, ai_agent_r_pop_sum),
    formattt(ai_prs_no_comments_count, ai_agent_r_sum),
    formattt(human_prs_no_comments_count, human_r_sum)
]
table2[("Review status", "$C_{AI}$")] = [
    formattt(pr_comments_agentic_comments_count, ai_agent_r_pop_sum),
    formattt(ai_comments_from_repos_with_human_agentic_comments_count, ai_agent_r_sum),
    formattt(human_comments_from_repos_with_human_agentic_comments_count, human_r_sum)
]
table2[("Review status", "$C_{human}$")] = [
    formattt(pr_comments_human_comments_count, ai_agent_r_pop_sum),
    formattt(ai_comments_from_repos_with_human_human_comments_count, ai_agent_r_sum),
    formattt(human_comments_from_repos_with_human_human_comments_count, human_r_sum)
]
table2[("Human review categories", "Evaluative")] = [
    formattt(pr_comments_df__human__counts["review"], ai_agent_r_pop_sum),
    formattt(pr_comments_from_repos_with_human_df__human__counts["review"], ai_agent_r_sum),
    formattt(human_pr_comments_from_repos_with_human_df__human__counts["review"], human_r_sum)
]
table2[("Human review categories", "Agentic")] = [
    formattt(pr_comments_df__human__counts["agentic_cmd"], ai_agent_r_pop_sum),
    formattt(pr_comments_from_repos_with_human_df__human__counts["agentic_cmd"], ai_agent_r_sum),
    formattt(human_pr_comments_from_repos_with_human_df__human__counts["agentic_cmd"], human_r_sum)
]
table2[("Human review categories", "Bot")] = [
    formattt(pr_comments_df__human__counts["bot"], ai_agent_r_pop_sum),
    formattt(pr_comments_from_repos_with_human_df__human__counts["bot"], ai_agent_r_sum),
    formattt(human_pr_comments_from_repos_with_human_df__human__counts["bot"], human_r_sum)
]
table2[("Human review categories", "Workflow")] = [
    formattt(pr_comments_df__human__counts["workflow"], ai_agent_r_pop_sum),
    formattt(pr_comments_from_repos_with_human_df__human__counts["workflow"], ai_agent_r_sum),
    formattt(human_pr_comments_from_repos_with_human_df__human__counts["workflow"], human_r_sum)
]
table2

<>:29: SyntaxWarning: invalid escape sequence '\%'
<>:29: SyntaxWarning: invalid escape sequence '\%'
C:\Users\julie\AppData\Local\Temp\ipykernel_9288\784244840.py:29: SyntaxWarning: invalid escape sequence '\%'
  return f"{x} " + f"({round(x * 100 / y, 2)}\%)"


Review status                                    \
  PR author Repository set            $C_0$         $C_{AI}$      $C_{human}$   
0  AI agent      $R_{pop}$  20621 (34.52\%)  28004 (46.87\%)  11118 (18.61\%)   
1  AI agent            $R$   2781 (10.75\%)  16727 (64.68\%)   6354 (24.57\%)   
2     Human            $R$   1924 (16.13\%)   5589 (46.87\%)    4412 (37.0\%)   

  Human review categories                                              
               Evaluative        Agentic           Bot       Workflow  
0           5767 (9.65\%)  3154 (5.28\%)  789 (1.32\%)  1408 (2.36\%)  
1          3333 (12.89\%)  1661 (6.42\%)   568 (2.2\%)   792 (3.06\%)  
2           3089 (25.9\%)    80 (0.67\%)  223 (1.87\%)  1020 (8.55\%)

In [64]:
print(table2.to_latex())

\begin{tabular}{llllllllll}
\toprule
 & \multicolumn{2}{r}{} & \multicolumn{3}{r}{Review status} & \multicolumn{4}{r}{Human review categories} \\
 & PR author & Repository set & $C_0$ & $C_{AI}$ & $C_{human}$ & Evaluative & Agentic & Bot & Workflow \\
\midrule
0 & AI agent & $R_{pop}$ & 20621 (34.52\%) & 28004 (46.87\%) & 11118 (18.61\%) & 5767 (9.65\%) & 3154 (5.28\%) & 789 (1.32\%) & 1408 (2.36\%) \\
1 & AI agent & $R$ & 2781 (10.75\%) & 16727 (64.68\%) & 6354 (24.57\%) & 3333 (12.89\%) & 1661 (6.42\%) & 568 (2.2\%) & 792 (3.06\%) \\
2 & Human & $R$ & 1924 (16.13\%) & 5589 (46.87\%) & 4412 (37.0\%) & 3089 (25.9\%) & 80 (0.67\%) & 223 (1.87\%) & 1020 (8.55\%) \\
\bottomrule
\end{tabular}



# Abstract table

## Reviewer Distribution

### pr_comments

In [65]:
bot_types = {"Bot", "BotOther"}

pr_user_groups = (
    pr_comments_df
    .assign(
        user_group=lambda df: df["user_type"].apply(
            lambda x: "Bot" if x in bot_types else "User"
        )
    )
    .groupby("pr_id")["user_group"]
    .apply(set)
)

agentic_only_prs = (pr_user_groups == {"Bot"}).sum()
human_only_prs = (pr_user_groups == {"User"}).sum()
mixed_prs = (pr_user_groups == {"Bot", "User"}).sum()

agentic_only_prs_perc = (agentic_only_prs * 100) / pr_comments_df["pr_id"].nunique()
human_only_prs_perc = (human_only_prs * 100) / pr_comments_df["pr_id"].nunique()
mixed_prs_perc = (mixed_prs * 100) / pr_comments_df["pr_id"].nunique()

total_prs_with_comments = pr_comments_df["pr_id"].nunique()

print(f"Agent only PRs: {agentic_only_prs} - {(agentic_only_prs_perc):.2f}%")
print(f"Human only PRs: {human_only_prs} - {(human_only_prs_perc):.2f}%")
print(f"Mixed PRs: {mixed_prs} - {(mixed_prs_perc):.2f}%")

Agent only PRs: 7958 - 61.33%
Human only PRs: 1066 - 8.22%
Mixed PRs: 3951 - 30.45%


#### Sanity Check

In [66]:
print(f"All unique PRs: \t\t\t{len(all_pr_ids)}")
print(f"Silent PRs: \t\t\t\t{len(prs_no_comments)}")
print(f"PRs with any comment: \t\t\t{len(all_pr_ids) - len(prs_no_comments)}\n")

print(f"All unique PRs with any comment: \t{pr_comments_df["pr_id"].nunique()}")
print(f"Sanity check sum: \t\t\t{agentic_only_prs + human_only_prs + mixed_prs}")
print(f"Sanity check sum (percent): \t\t{agentic_only_prs_perc + human_only_prs_perc + mixed_prs_perc}%")

All unique PRs: 			33596
Silent PRs: 				20621
PRs with any comment: 			12975

All unique PRs with any comment: 	12975
Sanity check sum: 			12975
Sanity check sum (percent): 		100.0%


### pr_comments_from_repos_with_human_df

In [67]:
bot_types = {"Bot", "BotOther"}

pr_user_groups_rwh = (
    pr_comments_from_repos_with_human_df
    .assign(
        user_group=lambda df: df["user_type"].apply(
            lambda x: "Bot" if x in bot_types else "User"
        )
    )
    .groupby("pr_id")["user_group"]
    .apply(set)
)

agentic_only_prs_rwh = (pr_user_groups_rwh == {"Bot"}).sum()
human_only_prs_rwh = (pr_user_groups_rwh == {"User"}).sum()
mixed_prs_rwh = (pr_user_groups_rwh == {"Bot", "User"}).sum()

agentic_only_prs_perc_rwh = (agentic_only_prs_rwh * 100) / pr_comments_from_repos_with_human_df["pr_id"].nunique()
human_only_prs_perc_rwh = (human_only_prs_rwh * 100) / pr_comments_from_repos_with_human_df["pr_id"].nunique()
mixed_prs_perc_rwh = (mixed_prs_rwh * 100) / pr_comments_from_repos_with_human_df["pr_id"].nunique()

total_prs_with_comments_rwh = pr_comments_from_repos_with_human_df["pr_id"].nunique()

print(f"Agent only PRs: {agentic_only_prs_rwh} - {(agentic_only_prs_perc_rwh):.2f}%")
print(f"Human only PRs: {human_only_prs_rwh} - {(human_only_prs_perc_rwh):.2f}%")
print(f"Mixed PRs: {mixed_prs_rwh} - {(mixed_prs_perc_rwh):.2f}%")

Agent only PRs: 3939 - 57.63%
Human only PRs: 552 - 8.08%
Mixed PRs: 2344 - 34.29%


#### Sanity Check

In [68]:
print(f"All unique PRs: \t\t\t{len(all_ai_prs)}")
print(f"Silent PRs: \t\t\t\t{len(ai_prs_no_comments)}")
print(f"PRs with any comment: \t\t\t{len(all_ai_prs) - len(ai_prs_no_comments)}\n")

print(f"All unique PRs with any comment: \t{pr_comments_from_repos_with_human_df["pr_id"].nunique()}")
print(f"Sanity check sum: \t\t\t{agentic_only_prs_rwh + human_only_prs_rwh + mixed_prs_rwh}")
print(f"Sanity check sum (percent): \t\t{agentic_only_prs_perc_rwh + human_only_prs_perc_rwh + mixed_prs_perc_rwh}%")

All unique PRs: 			9616
Silent PRs: 				2781
PRs with any comment: 			6835

All unique PRs with any comment: 	6835
Sanity check sum: 			6835
Sanity check sum (percent): 		100.0%


### human_pr_comments_from_repos_with_human_df

In [69]:
bot_types = {"Bot", "BotOther"}

pr_user_groups_hrwh = (
    human_pr_comments_from_repos_with_human_df
    .assign(
        user_group=lambda df: df["user_type"].apply(
            lambda x: "Bot" if x in bot_types else "User"
        )
    )
    .groupby("pr_id")["user_group"]
    .apply(set)
)

agentic_only_prs_hrwh = (pr_user_groups_hrwh == {"Bot"}).sum()
human_only_prs_hrwh = (pr_user_groups_hrwh == {"User"}).sum()
mixed_prs_hrwh = (pr_user_groups_hrwh == {"Bot", "User"}).sum()

agentic_only_prs_perc_hrwh = (agentic_only_prs_hrwh * 100) / human_pr_comments_from_repos_with_human_df["pr_id"].nunique()
human_only_prs_perc_hrwh = (human_only_prs_hrwh * 100) / human_pr_comments_from_repos_with_human_df["pr_id"].nunique()
mixed_prs_perc_hrwh = (mixed_prs_hrwh * 100) / human_pr_comments_from_repos_with_human_df["pr_id"].nunique()

total_prs_with_comments_hrwh = human_pr_comments_from_repos_with_human_df["pr_id"].nunique()

print(f"Agent only PRs: {agentic_only_prs_hrwh} - {(agentic_only_prs_perc_hrwh):.2f}%")
print(f"Human only PRs: {human_only_prs_hrwh} - {(human_only_prs_perc_hrwh):.2f}%")
print(f"Mixed PRs: {mixed_prs_hrwh} - {(mixed_prs_perc_hrwh):.2f}%")

Agent only PRs: 1932 - 52.93%
Human only PRs: 920 - 25.21%
Mixed PRs: 798 - 21.86%


#### Sanity Check

In [70]:
print(f"All unique PRs: \t\t\t{len(all_human_prs)}")
print(f"Silent PRs: \t\t\t\t{len(human_prs_no_comments)}")
print(f"PRs with any comment: \t\t\t{len(all_human_prs) - len(human_prs_no_comments)}\n")

print(f"All unique PRs with any comment: \t{human_pr_comments_from_repos_with_human_df["pr_id"].nunique()}")
print(f"Sanity check sum: \t\t\t{agentic_only_prs_hrwh + human_only_prs_hrwh + mixed_prs_hrwh}")
print(f"Sanity check sum (percent): \t\t{agentic_only_prs_perc_hrwh + human_only_prs_perc_hrwh + mixed_prs_perc_hrwh}%")

All unique PRs: 			5574
Silent PRs: 				1924
PRs with any comment: 			3650

All unique PRs with any comment: 	3650
Sanity check sum: 			3650
Sanity check sum (percent): 		100.0%


## Summary

### Functions

In [71]:
def summarize_pr_types(dataset_name, total_prs, agentic_only_prs, agentic_only_prs_perc, human_only_prs, human_only_prs_perc, mixed_prs, mixed_prs_perc):
    return {
        "Dataset": dataset_name,
        "PRs with comments": total_prs,
        "Agent only PRs": f"{agentic_only_prs} ({agentic_only_prs_perc:.2f}%)",
        "Human only PRs": f"{human_only_prs} ({human_only_prs_perc:.2f}%)",
        "Mixed PRs": f"{mixed_prs} ({mixed_prs_perc:.2f}%)",
    }

In [72]:
rows = []

rows.append(
    summarize_pr_types(
        "pr_comments",
        total_prs_with_comments,
        agentic_only_prs, agentic_only_prs_perc,
        human_only_prs, human_only_prs_perc,
        mixed_prs, mixed_prs_perc,
    )
)

rows.append(
    summarize_pr_types(
        "pr_comments_from_repos_with_human",
        total_prs_with_comments_rwh,
        agentic_only_prs_rwh, agentic_only_prs_perc_rwh,
        human_only_prs_rwh, human_only_prs_perc_rwh,
        mixed_prs_rwh, mixed_prs_perc_rwh,
    )
)

rows.append(
    summarize_pr_types(
        "human_pr_comments_from_repos_with_human",
        total_prs_with_comments_hrwh,
        agentic_only_prs_hrwh, agentic_only_prs_perc_hrwh,
        human_only_prs_hrwh, human_only_prs_perc_hrwh,
        mixed_prs_hrwh, mixed_prs_perc_hrwh,
    )
)

### Table

In [73]:
summary_df = pd.DataFrame(rows)
summary_df

,Dataset,PRs with comments,Agent only PRs,Human only PRs,Mixed PRs
0,pr_comments,12975,7958 (61.33%),1066 (8.22%),3951 (30.45%)
1,pr_comments_from_repos_with_human,6835,3939 (57.63%),552 (8.08%),2344 (34.29%)
2,human_pr_comments_from_repos_with_human,3650,1932 (52.93%),920 (25.21%),798 (21.86%)
